# Noisy curve centerline — pretrained ResNet34 U-Net

Run from top to bottom in **Kaggle Notebooks**, with **Notebook Settings → Accelerator → GPU**. This notebook trains on new synthetic images generated in memory. The synthetic image still contains a 2–5 px wide curve, dark grid, grain, baseline, and markers; the target has exactly **one labeled pixel per image column**. The model is a ResNet34 U-Net. The final trace uses the previous one-`y`-per-`x` dynamic program, not a 2D route finder.

The one-pixel label does **not** guarantee a one-pixel prediction. Inspect the probability map and the final trace. A single-valued `y(x)` cannot encode a genuinely vertical segment with several `y` values at the same `x`.

The notebook saves only checkpoints, metrics, and any CSV you explicitly export under `/kaggle/working`. It does **not** save a synthetic image dataset. An optional folder of manually labeled real scans can be used for validation; real images are never mixed into training here.


## 1. Install, imports and settings

Enable a GPU and Internet in the Kaggle Notebook settings. Internet is required to install `segmentation-models-pytorch` and download ImageNet encoder weights. Checkpoints are written to `/kaggle/working/curve_resnet34_centerline_v1`. Kaggle saves `/kaggle/working` as a **notebook version output** when you use **Save Version → Save & Run All**. `Quick Save` saves notebook edits, but does not execute training. For long runs, remember that an interactive session's files are temporary until captured in a saved version or downloaded.


In [ ]:
%pip -q install segmentation-models-pytorch opencv-python-headless scipy matplotlib tqdm

import csv, json, random, time
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import PchipInterpolator
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import segmentation_models_pytorch as smp

cv2.setNumThreads(0)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| device:', DEVICE)
if DEVICE.type != 'cuda':
    print('Enable a Colab GPU before full training.')

OUTPUT = Path('/kaggle/working/curve_resnet34_centerline_v1')
OUTPUT.mkdir(parents=True, exist_ok=True)
# To resume in a NEW Kaggle session, attach the previous notebook version as Input,
# set this to its output folder shown in Kaggle's Input pane, and set RESUME=True.
PREVIOUS_OUTPUT_DIR = ''  # example: '/kaggle/input/your-notebook-version/curve_resnet34_centerline_v1'
if PREVIOUS_OUTPUT_DIR:
    import shutil
    previous = Path(PREVIOUS_OUTPUT_DIR)
    for name in ('last.pt', 'best.pt', 'history.jsonl'):
        if (previous/name).is_file():
            shutil.copy2(previous/name, OUTPUT/name)
    print('Copied prior checkpoint files from', previous)

H, W = 224, 768             # training crop only; larger scans are tiled at inference
BATCH_SIZE = 16              # lower to 2 if Colab reports CUDA out of memory
SAMPLES_PER_EPOCH = 2000    # fresh synthetic images each epoch
VAL_SAMPLES = 200            # fixed synthetic validation images
TRACE_VAL_SAMPLES = 8       # subset used to evaluate the full tracer
EPOCHS = 100                # total epochs, including any resumed epochs
NUM_WORKERS = 2
LR = 3e-4
RESUME = True             # True resumes last.pt from this exact OUTPUT directory
MAX_STEP = 64              # same one-y-per-x DP constraint as before
JUMP_PENALTY = 0.01
REAL_VAL_DIR = ''           # optional: /kaggle/input/your-real-validation-dataset

print(f'{SAMPLES_PER_EPOCH} new training images per epoch; '
      f'{VAL_SAMPLES} fixed synthetic validation images; no synthetic dataset saved.')


## 2. Synthetic image generator

This follows the downloaded notebook's graph-paper, baseline, marker, spikes, grain and JPEG generator. Its rendered signal remains wide. The only label change is that `mask` contains one pixel at `round(y[x])` in each column. Generator code is placed directly in this notebook; no extra upload is required.


In [ ]:
# ============ synthetic chart generator (spikes, riding baseline, grain, grid) ============
"""Synthetic noisy strip-chart generator.

make_sample(H, W, rng) -> dict(
    image : float32 HxW in [0,1]        noisy chart (grain, grid, baseline, marker lines ...)
    mask  : uint8   HxW {0,1}           ONLY the signal curve (confusers are NOT labelled)
    poly  : float64 Wx2 [x, y]          exact ground-truth centre line, one point per column
)
Confusers = full-width thick baseline + full-height marker lines, drawn as dark as the
signal and sometimes touching/crossing it.  The net must learn *which* line is the signal.
Hard cases included on purpose: narrow deep SPIKES (V / merged-stroke / zig-zag clusters,
up and down), near-vertical edges, and curve RIDING the baseline.
"""
import cv2
import numpy as np
from scipy.interpolate import PchipInterpolator

SHIFT = 4  # cv2 fixed-point bits -> sub-pixel accurate drawing
_S = 1 << SHIFT


# ----------------------------------------------------------------------------- signal
def _signal_y(H, W, rng):
    """Random single-valued signal y(x), x = 0..W-1 (row coordinates, float)."""
    margin = 8
    center = rng.uniform(0.35, 0.65) * H
    step_std = rng.uniform(5, 22)          # typical knot-to-knot change (px)
    gap = rng.uniform(2.5, 14)             # typical knot spacing (px)

    xs = [0.0]
    while xs[-1] < W - 1:
        xs.append(xs[-1] + 1.0 + rng.gamma(2.0, gap / 2.0))
    xs = np.asarray(xs)
    xs[-1] = W - 1
    ys = np.empty(len(xs))
    ys[0] = center
    for i in range(1, len(xs)):
        step = rng.normal(0, step_std)
        if rng.random() < 0.08:            # steep jump / spike
            step *= rng.uniform(2.0, 4.0)
        ys[i] = ys[i - 1] + step + 0.06 * (center - ys[i - 1])

    x = np.arange(W, dtype=np.float64)
    y = PchipInterpolator(xs, ys)(x)

    for _ in range(rng.integers(0, 4)):    # smooth lobes
        c, w, a = rng.uniform(0, W), rng.uniform(15, 60), rng.uniform(-60, 60)
        y = y + a * np.exp(-0.5 * ((x - c) / w) ** 2)

    if rng.random() < 0.6:                 # flat lead-in (like the real trace start)
        n0 = int(rng.uniform(0.05, 0.45) * W)
        y[:n0] = y[n0]

    lo, hi = y.min(), y.max()              # fit inside image
    avail = H - 1 - 2 * margin
    if hi - lo > avail:
        y = (y - (hi + lo) / 2) * (avail / (hi - lo)) + (hi + lo) / 2
        lo, hi = y.min(), y.max()
    if lo < margin:
        y = y + (margin - lo)
    elif hi > H - 1 - margin:
        y = y - (hi - (H - 1 - margin))
    return y



def _one_spike(y, c, hw, sgn, frac, H, margin=8):
    """Add a triangular spike centred at integer column c (half-width hw px).
    frac = fraction of available room used as depth.  Returns y (new array)."""
    W = len(y)
    x = np.arange(W)
    lo, hi = max(0, int(c - hw - 3)), min(W, int(c + hw + 4))
    base = y[lo:hi]
    room = (H - 1 - margin - base.max()) if sgn > 0 else (base.min() - margin)
    if room < 14:
        return y
    shape = np.clip(1.0 - np.abs(x - c) / max(hw, 0.5), 0.0, 1.0)
    return y + sgn * frac * room * shape


def _add_spikes(y, H, rng):
    """Narrow deep spikes: V-shaped (strokes apart), merged (1-2 px wide), zig-zag clusters."""
    W = len(y)
    for _ in range(int(rng.choice([1, 1, 2, 2, 3, 4, 6]))):
        c = int(rng.integers(8, W - 8))
        hw = rng.uniform(0.5, 2.0) if rng.random() < 0.45 else rng.uniform(2.0, 9.0)
        sgn = 1 if rng.random() < 0.6 else -1
        y = _one_spike(y, c, hw, sgn, rng.uniform(0.3, 1.0), H)
        if rng.random() < 0.35:                      # zig-zag cluster around the spike
            for _ in range(int(rng.integers(1, 4))):
                c2 = int(np.clip(c + rng.integers(-22, 23), 3, W - 4))
                y = _one_spike(y, c2, rng.uniform(0.5, 4.0), sgn if rng.random() < 0.6 else -sgn,
                               rng.uniform(0.15, 0.7), H)
    return y


def _ride_baseline(y, by, rng):
    """Force a stretch of the signal to run along the baseline (touching / overlapping it)."""
    W = len(y)
    L = int(rng.integers(25, 150))
    a = int(rng.integers(0, max(1, W - L)))
    b = min(W, a + L)
    n = b - a
    off = rng.choice([-1, 1]) * rng.uniform(1.5, 5.0)
    w = np.ones(n)
    ramp = max(2, min(14, n // 3))
    w[:ramp] = np.linspace(0, 1, ramp)
    w[-ramp:] = np.linspace(1, 0, ramp)
    wobble = np.cumsum(rng.normal(0, 0.12, n))
    y = y.copy()
    y[a:b] = y[a:b] * (1 - w) + (by + off + wobble) * w
    return y


def _draw_poly(H, W, pts_xy, thickness, aa=True):
    canvas = np.zeros((H, W), np.uint8)
    p = np.round(np.asarray(pts_xy) * _S).astype(np.int32).reshape(1, -1, 2)
    cv2.polylines(canvas, p, False, 255, int(thickness),
                  cv2.LINE_AA if aa else cv2.LINE_8, shift=SHIFT)
    return canvas.astype(np.float32) / 255.0


# ----------------------------------------------------------------------------- paper
def _grid_layer(H, W, rng):
    """Darkness (0..1) of fine + major grid lines."""
    g = np.zeros((H, W), np.float32)
    strength = rng.uniform(0.15, 0.5)
    major_every = int(rng.choice([0, 4, 5, 5]))
    for axis, n in ((1, W), (0, H)):
        axis_gain = rng.uniform(0.9, 1.8) if axis == 1 else rng.uniform(0.6, 1.2)  # vertical lines darker
        period = rng.uniform(10, 32)
        pos, k = rng.uniform(0, period), 0
        while pos < n:
            i = int(round(pos))
            major = major_every and k % major_every == 0
            t = 2 if (major and rng.random() < 0.5) else 1
            v = strength * axis_gain * (1.4 if major else 1.0) * rng.uniform(0.7, 1.1)
            if axis == 1:
                g[:, i:i + t] = np.maximum(g[:, i:i + t], v)
            else:
                g[i:i + t, :] = np.maximum(g[i:i + t, :], v)
            pos += period
            k += 1
    return np.clip(g, 0, 0.9)


def _confusers(H, W, y, rng, force_by=None):
    """Full-width baseline + marker lines (all unlabelled)."""
    conf = np.zeros((H, W), np.float32)
    if force_by is not None or rng.random() < 0.85:          # baseline near the signal start
        by = force_by if force_by is not None else \
            y[: max(2, W // 10)].mean() + rng.choice([-1, 1]) * rng.uniform(0, 16)
        by = float(np.clip(by, 3, H - 4))
        t = int(rng.integers(2, 5))
        conf = np.maximum(conf, _draw_poly(H, W, [(0, by), (W - 1, by)], t))
    if rng.random() < 0.3:                                   # faint extra reference line
        by = rng.uniform(3, H - 4)
        conf = np.maximum(conf, 0.5 * _draw_poly(H, W, [(0, by), (W - 1, by)], int(rng.integers(1, 3))))
    for _ in range(int(rng.choice([0, 1, 1, 2]))):           # vertical marker(s)
        vx = rng.uniform(20, W - 20)
        conf = np.maximum(conf, _draw_poly(H, W, [(vx, 0), (vx, H - 1)], int(rng.integers(2, 7))))
    return conf


# ----------------------------------------------------------------------------- sample
def make_sample(H=224, W=768, rng=None, spike_prob=0.8, ride_prob=0.35):
    rng = rng if rng is not None else np.random.default_rng()
    y = _signal_y(H, W, rng)
    if rng.random() < spike_prob:
        y = _add_spikes(y, H, rng)
    force_by = None
    if rng.random() < ride_prob:                             # curve rides / touches the baseline
        force_by = float(np.clip(y[int(rng.integers(0, W))], 6, H - 7))
        y = _ride_baseline(y, force_by, rng)
    x = np.arange(W, dtype=np.float64)
    poly = np.stack([x, y], 1)

    t_sig = int(rng.integers(2, 6))                          # 2..5 px ink width
    sig = _draw_poly(H, W, poly, t_sig)
    # Exactly one target pixel per x; sig stays 2-5 px wide in the input.
    mask = np.zeros((H, W), np.uint8)
    mask[np.clip(np.rint(y).astype(np.int32), 0, H - 1), np.arange(W)] = 1
    conf = _confusers(H, W, y, rng, force_by)
    ink = np.maximum(sig, conf)

    # paper
    img = np.full((H, W), rng.uniform(0.82, 0.97), np.float32)
    low = cv2.GaussianBlur(rng.normal(0, 1, (H, W)).astype(np.float32), (0, 0), rng.uniform(3, 10))
    img += low / (low.std() + 1e-6) * rng.uniform(0.0, 0.03)
    img *= 1.0 - _grid_layer(H, W, rng)

    # ink
    a, dark = rng.uniform(0.75, 0.97), rng.uniform(0.03, 0.2)
    img = img * (1 - a * ink) + a * ink * dark
    drop = (rng.random((H, W)) < rng.uniform(0, 0.12)) * ink * rng.uniform(0.3, 0.7)
    img += drop.astype(np.float32)                           # patchy ink

    # grain speckle + gaussian noise
    dens = rng.uniform(0.04, 0.5)
    speck = (rng.random((H, W)) < dens) * rng.uniform(0.15, 0.6, (H, W))
    img *= (1.0 - speck).astype(np.float32)
    img += rng.normal(0, rng.uniform(0.01, 0.08), (H, W)).astype(np.float32)

    s = rng.uniform(0, 1.1)
    if s > 0.25:
        img = cv2.GaussianBlur(img, (0, 0), s)
    img = np.clip(img, 0, 1) ** rng.uniform(0.8, 1.3)
    img = (img - 0.5) * rng.uniform(0.8, 1.2) + 0.5 + rng.uniform(-0.05, 0.05)
    img = np.clip(img, 0, 1).astype(np.float32)

    if rng.random() < 0.5:                                   # jpeg artefacts
        q = int(rng.integers(35, 95))
        ok, buf = cv2.imencode(".jpg", (img * 255).astype(np.uint8), [cv2.IMWRITE_JPEG_QUALITY, q])
        if ok:
            img = cv2.imdecode(buf, cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0

    return {"image": img, "mask": mask, "poly": poly}



## 3. Centerline-safe augmentation and on-demand datasets

The downloaded notebook warped a thick mask and thresholded it. That could erase a one-pixel target. Here horizontal/vertical scale, translation and flips are applied to both image and analytic curve coordinates; **the one-pixel label is rasterized after the transform**. Rotation is omitted because it can turn a steep `y(x)` into a multi-valued curve.


In [ ]:
def centerline_mask(y, height):
    y = np.asarray(y, np.float32)
    w = len(y)
    mask = np.zeros((height, w), np.float32)
    valid = np.isfinite(y) & (y >= 0) & (y < height)
    cols = np.flatnonzero(valid)
    mask[np.rint(y[valid]).astype(np.int32).clip(0, height - 1), cols] = 1.0
    return mask, valid.astype(np.float32)

def augment_centerline(img, poly, rng):
    h, w = img.shape
    y = np.asarray(poly[:, 1], np.float32)
    # X-preserving affine warp: x'=sx*x+tx, y'=sy*y+ty.
    sx, sy = rng.uniform(.85, 1.18), rng.uniform(.82, 1.22)
    tx = (w-1)*(1-sx)/2 + rng.uniform(-12, 12)
    ty = (h-1)*(1-sy)/2 + rng.uniform(-10, 10)
    M = np.array([[sx, 0, tx], [0, sy, ty]], np.float32)
    fill = float(np.median(img))
    out = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR,
                         borderMode=cv2.BORDER_CONSTANT, borderValue=fill)
    dest_x = np.arange(w, dtype=np.float32)
    src_x = (dest_x-tx)/sx
    out_y = sy*np.interp(np.clip(src_x, 0, w-1), np.arange(w), y)+ty
    out_y[(src_x < 0) | (src_x > w-1)] = np.nan
    if rng.random() < .5:
        out = np.ascontiguousarray(out[:, ::-1]); out_y = out_y[::-1].copy()
    if rng.random() < .3:
        out = np.ascontiguousarray(out[::-1]); out_y = h-1-out_y
    if rng.random() < .7:
        out = np.clip(out, 0, 1) ** rng.uniform(.75, 1.35)
        out = (out-.5)*rng.uniform(.75, 1.25)+.5+rng.uniform(-.08, .08)
    if rng.random() < .4:
        out *= 1-(rng.random(out.shape)<rng.uniform(.02, .15))*rng.uniform(.1, .5)
    if rng.random() < .5:
        out += rng.normal(0, rng.uniform(.005, .06), out.shape)
    if rng.random() < .3:
        out = cv2.GaussianBlur(out.astype(np.float32), (0, 0), rng.uniform(.3, 1.2))
    return np.clip(out, 0, 1).astype(np.float32), out_y

MEAN, STD = .45, .226       # same grayscale normalization as downloaded notebook
class OnlineCurves(Dataset):
    def __init__(self, count, offset, training=False):
        self.count, self.offset, self.training, self.epoch = count, offset, training, 0
    def __len__(self): return self.count
    def __getitem__(self, i):
        seed = self.offset + (self.epoch*self.count if self.training else 0) + i
        rng = np.random.default_rng(seed)
        sample = make_sample(H, W, rng)
        img, y = sample['image'], sample['poly'][:, 1]
        if self.training:
            img, y = augment_centerline(img, sample['poly'], rng)
        target, valid = centerline_mask(y, H)
        return (torch.from_numpy(((img-MEAN)/STD).astype(np.float32))[None],
                torch.from_numpy(target)[None],
                torch.from_numpy(valid.astype(np.float32)),
                torch.from_numpy(np.asarray(y, np.float32).copy()))

train_ds = OnlineCurves(SAMPLES_PER_EPOCH, 0, training=True)
val_ds = OnlineCurves(VAL_SAMPLES, 10_000_000, training=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                          shuffle=False, persistent_workers=False,
                          pin_memory=DEVICE.type == 'cuda')
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, num_workers=0)


## 4. Preview before training

Check that the **red dots** sit on the visible ink through grid crossings and narrow dips. The white label has one pixel per column. If the synthetic scans still do not resemble your real scans, adjust the generator before training.


In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 9))
for row, seed in enumerate([100, 101, 102, 103]):
    s = make_sample(H, W, np.random.default_rng(seed))
    image, mask, poly = s['image'], s['mask'], s['poly']
    axes[row, 0].imshow(image, cmap='gray', vmin=0, vmax=1)


    axes[row, 1].plot(poly[:, 0], poly[:, 1], color="white", linewidth=1)
# axes[row, 1].set_xlim(0, W - 1)
# axes[row, 1].set_ylim(H - 1, 0)
# axes[row, 1].set_facecolor("black")
    
    axes[row, 0].plot(poly[:,0], poly[:,1], '.', color='red', markersize=.5)
    axes[row, 0].set_title(f'Synthetic scan {seed}: wide ink, red centerline')
    axes[row, 1].imshow(mask, cmap='gray', vmin=0, vmax=1)
    axes[row, 1].set_title(f'One-pixel target; {int(mask.sum())} pixels / {W} columns')
    for ax in axes[row]: ax.axis('off')
plt.tight_layout(); plt.show()

sample = train_ds[0]
assert sample[1].sum(1).max().item() <= 1, 'More than one target pixel in a column'
print('Augmented training target: one or zero pixel per column; valid columns:',
      int(sample[2].sum()), '/', W)


## 5. ResNet34 U-Net and one-pixel loss

The architecture is the downloaded notebook's `smp.Unet` with ImageNet ResNet34 encoder and one grayscale input. It **requires** pretrained weights; it will fail clearly if they cannot download. The loss changes because a one-pixel target is much sparser than the downloaded notebook's wide mask: column-wise cross-entropy rewards the correct row, while weighted BCE and Dice suppress other ink. The downloaded clDice and stroke-height steep weighting are omitted because they were designed for thick connected masks. Steep columns here are detected from the known centerline coordinates.


In [ ]:
model = smp.Unet(encoder_name='resnet34', encoder_weights='imagenet',
                 in_channels=1, classes=1, activation=None).to(DEVICE)
print('ResNet34 U-Net parameters:', sum(p.numel() for p in model.parameters()))
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')

def centerline_loss(logits, target, valid, y):
    logits = logits.float(); target = target.float()
    valid = valid.float()
    y = y.float()
    # Adjacent target-row changes indicate sharp turns and narrow peaks/dips.
    dy = torch.abs(y[:, 1:] - y[:, :-1])
    dy = torch.nan_to_num(dy, nan=0.0, posinf=0.0, neginf=0.0)
    steep = F.pad((dy > 5).float(), (1, 0))
    steep = F.max_pool1d(steep[:, None], kernel_size=7, stride=1, padding=3)[:, 0]
    col_weight = valid*(1+2*steep)
    # Softmax across image rows teaches one row per valid column.
    log_rows = F.log_softmax(logits[:, 0], dim=1)
    ce = -(log_rows*target[:, 0]).sum(dim=1)
    ce = (ce*col_weight).sum()/col_weight.sum().clamp_min(1)
    pixel_weight = 1+19*target*(1+2*steep[:, None, None, :])
    bce = F.binary_cross_entropy_with_logits(logits, target, weight=pixel_weight)
    p = torch.sigmoid(logits)
    dice = 1-(2*(p*target).sum((1,2,3))+1)/(p.sum((1,2,3))+target.sum((1,2,3))+1)
    return .5*ce + bce + dice.mean()

def pixel_dice(logits, target):
    p = (logits.sigmoid() > .5).float()
    return ((2*(p*target).sum((1,2,3))+1)/
            (p.sum((1,2,3))+target.sum((1,2,3))+1)).mean().item()


## 6. Probability map and original one-y-per-x tracer

Inference uses overlapping native-resolution tiles. The dynamic program chooses exactly one row in each image column, with the same `max_step=64` and `penalty=.01` defaults as the previous notebook. If a model assigns high probability to a grid line, the tracker may still follow it: inspect the map before blaming the tracker.


In [ ]:
@torch.inference_mode()
def predict_tiled(scan, tile_h=H, tile_w=W):
    model.eval()
    scan = np.asarray(scan)
    if scan.ndim != 2: raise ValueError('Pass one grayscale scan/crop')
    if scan.dtype == np.uint8: scan = scan.astype(np.float32)/255.0
    else: scan = scan.astype(np.float32)
    h, w = scan.shape
    padded = np.pad(scan, ((0, max(0,tile_h-h)), (0,max(0,tile_w-w))), mode='edge')
    ph, pw = padded.shape
    def starts(n, t): return sorted(set(list(range(0,n-t+1,max(1,t//2)))+[n-t]))
    accum = np.zeros((ph,pw), np.float32)
    weight = np.zeros((ph,pw), np.float32)
    window = np.maximum(.05, np.outer(np.hanning(tile_h),np.hanning(tile_w))).astype(np.float32)
    for top in starts(ph,tile_h):
        for left in starts(pw,tile_w):
            patch = padded[top:top+tile_h,left:left+tile_w]
            tensor = torch.from_numpy(((patch-MEAN)/STD).copy())[None,None].to(DEVICE)
            with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type=='cuda'):
                p = model(tensor).sigmoid()[0,0].float().cpu().numpy()
            accum[top:top+tile_h,left:left+tile_w] += p*window
            weight[top:top+tile_h,left:left+tile_w] += window
    return (accum/weight)[:h,:w]

def trace(probability, max_step=MAX_STEP, penalty=JUMP_PENALTY):
    h, w = probability.shape
    cost = -np.log(np.clip(probability, 1e-6, 1))
    dp = cost[:,0].copy()
    parent = np.zeros((h,w), np.int32)
    rows = np.arange(h)
    offsets = np.arange(-min(max_step,h-1),min(max_step,h-1)+1)
    predecessors = rows[:,None]+offsets[None,:]
    valid = (predecessors>=0)&(predecessors<h)
    clipped = np.clip(predecessors,0,h-1)
    for x in range(1,w):
        scores = dp[clipped]+penalty*np.abs(offsets)[None,:]
        scores[~valid] = np.inf
        choice = scores.argmin(axis=1)
        parent[:,x] = clipped[rows,choice]
        dp = cost[:,x]+scores[rows,choice]
    y = np.empty(w,np.int32); y[-1] = dp.argmin()
    for x in range(w-1,0,-1): y[x-1] = parent[y[x],x]
    return y

def preview_trace(image, probability, y, title=''):
    fig, axes = plt.subplots(3,1,figsize=(16,8),sharex=True)
    axes[0].imshow(image,cmap='gray',vmin=0,vmax=1); axes[0].set_title(title or 'Input')
    axes[1].imshow(probability,cmap='magma',vmin=0,vmax=1); axes[1].set_title('ResNet34 probability')
    axes[2].imshow(image,cmap='gray',vmin=0,vmax=1)
    axes[2].plot(np.arange(len(y)),y,'r',lw=.8)
    axes[2].set_title('One-y-per-x trace')
    plt.tight_layout(); plt.show()


## 7. Optional real validation

Leave `REAL_VAL_DIR=''` in settings to skip this. To validate on real images, place matching files as an attached Kaggle Dataset under `/kaggle/input`:

```text
/kaggle/input/your-real-validation-dataset/
  images/scan_01.png
  labels/scan_01.csv
```

The CSV must contain pixel coordinates `x,y` for the **same crop**, ideally one row per image column from `x=0` to `width-1`. Example: `0,212`, `1,212`, `2,213`. Keep final test scans separate. The first plotted overlay lets you check label alignment.


In [ ]:
def load_real_pairs(directory):
    if not directory: return []
    root = Path(directory); image_dir = root/'images'; label_dir = root/'labels'
    if not image_dir.is_dir() or not label_dir.is_dir():
        raise FileNotFoundError(f'Expected {image_dir} and {label_dir}')
    pairs = []
    for path in sorted(image_dir.iterdir()):
        if path.suffix.lower() not in ('.png','.jpg','.jpeg','.tif','.tiff','.bmp'): continue
        label = label_dir/(path.stem+'.csv')
        if not label.is_file(): raise FileNotFoundError(label)
        img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if img is None: raise ValueError(f'Cannot read {path}')
        points = []
        with label.open(newline='') as f:
            for row in csv.reader(f):
                try: points.append((float(row[0]),float(row[1])))
                except (ValueError,IndexError): continue
        points = np.asarray(points,np.float32)
        if len(points)<2: raise ValueError(f'Need x,y coordinates in {label}')
        if np.any(points[:,0]<0)|np.any(points[:,0]>=img.shape[1])|np.any(points[:,1]<0)|np.any(points[:,1]>=img.shape[0]):
            raise ValueError(f'Label outside image: {label}')
        pairs.append((path.name,img.astype(np.float32)/255,points))
    if not pairs: raise ValueError(f'No real images in {image_dir}')
    return pairs

real_pairs = load_real_pairs(REAL_VAL_DIR)
print('Real validation images:',len(real_pairs))
if real_pairs:
    name,img,points = real_pairs[0]
    plt.figure(figsize=(16,4)); plt.imshow(img,cmap='gray',vmin=0,vmax=1)
    plt.plot(points[:,0],points[:,1],'r',lw=.8)
    plt.title('Verify manual label alignment: '+name); plt.show()

def reference_y(points, width):
    order = np.argsort(points[:,0]); x=points[order,0]; y=points[order,1]
    if np.any(np.diff(x)<=0): raise ValueError('Real validation x must be strictly increasing')
    cols = np.arange(width); inside=(cols>=x[0])&(cols<=x[-1])
    ref = np.full(width,np.nan,np.float32)
    ref[inside]=np.interp(cols[inside],x,y)
    return ref

def trace_metrics(pred, ref):
    ok=np.isfinite(ref)
    if not ok.any(): return {'mae_px':float('nan'),'within_2px':float('nan')}
    err=np.abs(pred[ok]-ref[ok])
    return {'mae_px':float(err.mean()),'within_2px':float((err<=2).mean())}


## 8. Train / resume

Each epoch samples `SAMPLES_PER_EPOCH` new synthetic images. `last.pt` is saved each completed epoch and `best.pt` is selected by **real validation within 2 px** if real labels are provided, otherwise by **synthetic trace within 2 px**. The metric is one-y-per-x column accuracy, not a wide-mask Dice score. `RESUME=True` loads model, optimizer, scaler and current epoch. In a new Kaggle session, attach the previous notebook version as Input, set `PREVIOUS_OUTPUT_DIR` in cell 1 and `RESUME=True`, then rerun top to bottom. Keep the same configuration. Stop a Colab cell using the square stop button; only completed epochs have checkpoints.


In [ ]:
def save_checkpoint(path, epoch, best_score):
    torch.save({'epoch':epoch,'model':model.state_dict(),'optimizer':optimizer.state_dict(),
                'scaler':scaler.state_dict(),'best_score':best_score,
                'config':{'H':H,'W':W,'batch_size':BATCH_SIZE,'samples_per_epoch':SAMPLES_PER_EPOCH,
                          'target':'one_pixel_per_column','encoder':'resnet34'}},path)

start_epoch, best_score = 1, -float('inf')
if RESUME:
    ckpt_path = OUTPUT/'last.pt'
    if not ckpt_path.is_file(): raise FileNotFoundError(ckpt_path)
    ckpt = torch.load(ckpt_path,map_location=DEVICE,weights_only=False)
    if ckpt.get('config',{}).get('target')!='one_pixel_per_column':
        raise ValueError('Checkpoint target is incompatible with this notebook')
    model.load_state_dict(ckpt['model']); optimizer.load_state_dict(ckpt['optimizer'])
    scaler.load_state_dict(ckpt['scaler'])
    start_epoch = ckpt['epoch']+1; best_score = ckpt['best_score']
    print('Resuming from completed epoch',ckpt['epoch'])

for epoch in range(start_epoch,EPOCHS+1):
    train_ds.epoch = epoch-1
    model.train(); total=0.0; count=0; t0=time.time()
    for images,targets,valid,yy in tqdm(train_loader,desc=f'Epoch {epoch}/{EPOCHS}'):
        images=images.to(DEVICE,non_blocking=True); targets=targets.to(DEVICE,non_blocking=True)
        valid=valid.to(DEVICE,non_blocking=True); yy=yy.to(DEVICE,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type,enabled=DEVICE.type=='cuda'):
            logits=model(images)
        loss=centerline_loss(logits,targets,valid,yy)
        scaler.scale(loss).backward(); scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(),5)
        scaler.step(optimizer); scaler.update()
        total+=loss.item()*len(images); count+=len(images)
    model.eval(); val_loss=0.0; val_dice=0.0; vn=0
    with torch.inference_mode():
        for images,targets,valid,yy in val_loader:
            images=images.to(DEVICE);targets=targets.to(DEVICE)
            valid=valid.to(DEVICE);yy=yy.to(DEVICE)
            logits=model(images)
            val_loss+=centerline_loss(logits,targets,valid,yy).item()*len(images)
            val_dice+=pixel_dice(logits,targets)*len(images);vn+=len(images)
    synth_scores=[]
    for i in range(min(TRACE_VAL_SAMPLES,VAL_SAMPLES)):
        sample=make_sample(H,W,np.random.default_rng(10_000_000+i))
        p=predict_tiled(sample['image'])
        pred=trace(p)
        synth_scores.append(trace_metrics(pred,sample['poly'][:,1]))
    synth_within=float(np.mean([m['within_2px'] for m in synth_scores]))
    synth_mae=float(np.mean([m['mae_px'] for m in synth_scores]))
    real_scores=[]
    for name,img,points in real_pairs:
        p=predict_tiled(img); pred=trace(p)
        real_scores.append(trace_metrics(pred,reference_y(points,img.shape[1])))
    real_within=float(np.mean([m['within_2px'] for m in real_scores])) if real_scores else None
    score=real_within if real_scores else synth_within
    save_checkpoint(OUTPUT/'last.pt',epoch,max(best_score,score))
    if score>best_score:
        best_score=score;save_checkpoint(OUTPUT/'best.pt',epoch,best_score)
    record={'epoch':epoch,'train_loss':total/count,'val_loss':val_loss/vn,
            'val_pixel_dice':val_dice/vn,'synth_within_2px':synth_within,
            'synth_mae_px':synth_mae,'real_within_2px':real_within,
            'seconds':round(time.time()-t0,1)}
    with (OUTPUT/'history.jsonl').open('a') as f: f.write(json.dumps(record)+'\n')
    print(record,flush=True)

print('Training cell finished. Best checkpoint:',OUTPUT/'best.pt')


## 9. Inspect held-out synthetic and your real scan

This loads `best.pt` from `/kaggle/working`, so it works after training reaches a completed epoch. The held-out seed is separate from training and validation. To check your own scan, attach it as a Kaggle Dataset, then set `IMAGE_PATHS` in the last cell to the path shown in Kaggle's Input pane. The CSV contains one `(x,y)` per image column, in **crop pixel coordinates**. If the probability map misses a segment, trace tuning alone will not restore it.


In [ ]:
ckpt=torch.load(OUTPUT/'best.pt',map_location=DEVICE,weights_only=False)
model.load_state_dict(ckpt['model']);model.eval()
print('Loaded best epoch:',ckpt['epoch'])
s=make_sample(H,W,np.random.default_rng(20_000_000))
p=predict_tiled(s['image']); y=trace(p)
print('Held-out synthetic:',trace_metrics(y,s['poly'][:,1]))
preview_trace(s['image'],p,y,'Held-out synthetic image')


In [ ]:
import hashlib

# Attach your own image dataset to the Kaggle notebook, then add its paths.
IMAGE_PATHS = []  # e.g. ['/kaggle/input/my-scans/scan_01.png']

for image_path in IMAGE_PATHS:
    path = Path(image_path)
    scan = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if scan is None:
        raise FileNotFoundError(path)

    image = scan.astype(np.float32) / 255.0
    probability = predict_tiled(image)
    y = trace(probability)

    # Separate folder per input, even if two inputs have the same filename.
    key = hashlib.sha1(str(path).encode("utf-8")).hexdigest()[:8]
    result_dir = OUTPUT / "predictions" / f"{path.stem}_{key}"
    result_dir.mkdir(parents=True, exist_ok=True)

    # Full-resolution scan with the traced curve drawn in red.
    overlay = cv2.cvtColor(scan, cv2.COLOR_GRAY2BGR)
    points = np.column_stack((
        np.arange(len(y), dtype=np.int32),
        y.astype(np.int32)
    )).reshape(-1, 1, 2)
    cv2.polylines(
        overlay, [points], isClosed=False,
        color=(0, 0, 255), thickness=1, lineType=cv2.LINE_AA
    )

    overlay_path = result_dir / "trace_overlay.png"
    if not cv2.imwrite(str(overlay_path), overlay):
        raise OSError(f"Could not save {overlay_path}")

    # Viewable probability image: black=0, white=1.
    probability_8bit = np.rint(
        np.clip(probability, 0, 1) * 255
    ).astype(np.uint8)
    probability_path = result_dir / "probability.png"
    if not cv2.imwrite(str(probability_path), probability_8bit):
        raise OSError(f"Could not save {probability_path}")

    # Exact floating-point probabilities for later analysis.
    np.save(result_dir / "probability.npy", probability.astype(np.float32))

    csv_path = result_dir / "trace.csv"
    with csv_path.open("w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["x", "y"])
        writer.writerows(zip(range(len(y)), y.tolist()))

    preview_trace(image, probability, y, path.name)
    plt.close("all")

    print(f"\n{path.name}")
    print("Trace overlay:   ", overlay_path)
    print("Probability map: ", probability_path)
    print("Exact values:    ", result_dir / "probability.npy")
    print("Coordinates:     ", csv_path)

In [ ]:
# Optional: archive inference results for download.
if (OUTPUT / 'predictions').is_dir():
    import shutil
    archive = shutil.make_archive(str(OUTPUT / 'predictions'), 'zip', root_dir=OUTPUT, base_dir='predictions')
    print(archive)
else:
    print('Run inference first; there are no predictions to archive.')
